In [1]:
import requests
import pandas as pd
import time
from tqdm import tqdm
from google.colab import files

##############################################################
# USER SETTINGS & CREDENTIALS
##############################################################

ELSEVIER_API_KEY = "7f34bdaaf30d51ab3d181537aa644f7c"
INST_TOKEN = ""  # Optional: Add institutional token here if provided by Makerere library

# Comprehensive Scopus search query matching your thesis scope
SCOPUS_QUERY = (
    'TITLE-ABS-KEY(('
    '"honey bee*" OR "honeybee*" OR "Apis" OR "Apis mellifera" OR "Apis cerana" OR "Apis dorsata" OR '
    '"stingless bee*" OR "Meliponini" OR "Melipona" OR "Meliponula" OR "bumblebee*" OR "Bombus" OR '
    '"Bombus terrestris" OR "Bombus impatiens" OR "non-Apis" OR "wild bee*" OR "pollinator*" OR "Apoidea"'
    ') AND ('
    '"pesticide*" OR "agrochemical*" OR "insecticide*" OR "neonicotinoid*" OR "imidacloprid" OR '
    '"thiamethoxam" OR "clothianidin" OR "acetamiprid" OR "thiacloprid" OR "flupyradifurone" OR '
    '"fungicide*" OR "herbicide*" OR "glyphosate" OR "acaricide*" OR "fipronil" OR "organophosphate*" OR '
    '"pyrethroid*" OR "carbamate*" OR "chlorpyrifos" OR "deltamethrin"'
    ') AND ('
    '"transcriptom*" OR "RNA-Seq" OR "RNA sequencing" OR "gene expression" OR "differentially expressed" OR '
    '"DEG" OR "DEGs" OR "microarray*" OR "proteom*" OR "metabolom*" OR "multi-omics" OR "multiomics" OR '
    '"ecotoxicogenom*" OR "toxicogenom*" OR "next generation sequencing" OR "NGS"'
    ')) AND PUBYEAR AFT 2009 AND PUBYEAR BEF 2027'
)

url = "https://api.elsevier.com/content/search/scopus"

headers = {
    "X-ELS-APIKey": ELSEVIER_API_KEY,
    "Accept": "application/json"
}
if INST_TOKEN:
    headers["X-ELS-Insttoken"] = INST_TOKEN

params = {
    "query": SCOPUS_QUERY,
    "count": 25,  # Elsevier Scopus API maximum batch size per call
    "start": 0
}

##############################################################
# CONNECTION & COUNT TEST
##############################################################

print("Connecting to Elsevier Scopus API...")
r = requests.get(url, headers=headers, params=params)

if r.status_code == 401:
    print("\n❌ 401 Unauthorized Error: Key rejected by Elsevier.")
    print("Response:", r.text)
    r.raise_for_status()

r.raise_for_status()
data = r.json()

total_results = int(data["search-results"]["opensearch:totalResults"])
print(f"Connected successfully!")
print(f"Total Scopus records found (2010–2026): {total_results}\n")

##############################################################
# BATCH EXTRACTION
##############################################################

records = []
batch_size = 25
max_records = min(total_results, 5000)  # Fetch up to 5,000 matches

for start in tqdm(range(0, max_records, batch_size), desc="Downloading Scopus Metadata"):
    params["start"] = start
    resp = requests.get(url, headers=headers, params=params)

    if resp.status_code == 200:
        entries = resp.json().get("search-results", {}).get("entry", [])

        for entry in entries:
            # Extract DOI
            doi = entry.get("prism:doi", "")

            # Extract Title
            title = entry.get("dc:title", "")

            # Extract Authors
            authors = entry.get("dc:creator", "")

            # Extract Journal Name
            journal = entry.get("prism:publicationName", "")

            # Extract Publication Year
            cover_date = entry.get("prism:coverDate", "")
            pub_year = cover_date[:4] if cover_date and len(cover_date) >= 4 else ""

            # Extract Abstract (if provided by API payload)
            abstract = entry.get("dc:description", "")

            # Extract Scopus URL / EID
            eid = entry.get("eid", "")
            scopus_link = entry.get("link", [{}, {}])[0].get("@href", "") if entry.get("link") else ""

            records.append({
                "EID": eid,
                "DOI": doi,
                "Title": title,
                "Authors": authors,
                "Journal": journal,
                "Year": pub_year,
                "Abstract": abstract,
                "Scopus_URL": scopus_link
            })

    time.sleep(0.3)  # Respect Elsevier API rate limits

##############################################################
# EXPORT TO EXCEL
##############################################################

df = pd.DataFrame(records)

# Drop any duplicate records by DOI
df = df.drop_duplicates(subset=["DOI"], keep="first")

filename = "Scopus_Bee_Toxicogenomics_2010_2026.xlsx"
df.to_excel(filename, index=False, engine='openpyxl')

print(f"\nExtraction Complete! Downloaded {len(df)} unique Scopus records.")
print(df.head(3))

# Trigger automatic browser download in Colab
files.download(filename)

Connecting to Elsevier Scopus API...
Connected successfully!
Total Scopus records found (2010–2026): 634




Extraction Complete! Downloaded 631 unique Scopus records.
                   EID                         DOI  \
0  2-s2.0-105027735505  10.1186/s12302-025-01280-w   
1  2-s2.0-105035440468  10.1038/s41598-026-42605-w   
2  2-s2.0-105038461319  10.1186/s12864-026-12770-3   

                                               Title       Authors  \
0  Synergistic impacts of flupyradifurone and uni...       Chen L.   
1  Metagenomic and gene expression patterns in de...    Nearman A.   
2  The diversity of insecticide target site mutat...  Verbinnen G.   

                         Journal  Year Abstract  \
0  Environmental Sciences Europe  2026            
1             Scientific Reports  2026            
2                   BMC Genomics  2026            

                                          Scopus_URL  
0  https://api.elsevier.com/content/abstract/scop...  
1  https://api.elsevier.com/content/abstract/scop...  
2  https://api.elsevier.com/content/abstract/scop...  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>